In [1]:
import numpy as np

import pandas as pd
pd.options.plotting.backend = "plotly"

import os
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
import scipy
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.filterwarnings('once')

In [2]:
import tensorflow as tf
tf.__version__

'2.17.0'

# Processamento e seleção da melhor coluna

In [3]:
best_column = 'NDCG@10'
metric_type = ["Prec", "Rec", "F1_Score", "Hit_Rate", "NDCG"]
top_k = [3, 5, 10, 20]

main_path = "results/metrics"
main_file = os.listdir(main_path)

dataframes = []

for dataset_name in main_file:
        
    recommender_files = os.listdir(os.path.join(main_path, dataset_name))
    for recommender_name in recommender_files:
                
        metrics_path = os.path.join("results", "metrics", dataset_name, recommender_name, "metrics.csv")
        metrics_aux = pd.read_csv(metrics_path, sep=';')
        
        metrics_aux.insert(0,'Recommender','')
        metrics_aux['Recommender'] = recommender_name

        metrics_aux.insert(0,'Dataset','')
        metrics_aux['Dataset'] = dataset_name
        
        dataframes.append(metrics_aux)
                
metrics_df = pd.concat(dataframes, ignore_index=True)

if (best_column == 'Mean'):
    metrics_df.insert(3,'Mean', metrics_df.mean(axis=1))
    
if best_column in metrics_df.columns:
    saida = metrics_df.loc[metrics_df.reset_index().groupby(['Dataset', 'Recommender'])[best_column].idxmax()].reset_index(drop=True)
else:
    raise Exception("Coluna '" + best_column + "' não encontrada")

In [5]:
pd.options.display.max_colwidth = 100

df_novo = metrics_df[metrics_df["Recommender"] == "Item2Vec_itemSim"]
df_novo['epoch'] = df_novo['Parameters'].str.split('-').str[-1]
#df_novo = df_novo[df_novo["Dataset"] == "BestBuy"]
df_novo = df_novo[df_novo["epoch"] == "140"]

my_regex = "Recommender|Parameters|NDCG@20"
df_novo = df_novo.filter(regex=(my_regex))
display(df_novo)

C:\Users\Rafael Tofoli\AppData\Local\Temp\ipykernel_12372\1839612541.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_novo['epoch'] = df_novo['Parameters'].str.split('-').str[-1]


,Recommender,Parameters,NDCG@20
110,Item2Vec_itemSim,epochs-160_learning_rate-0.25_negative_exp--0.5_subsample-0.001_w_size--1_epochs-140,0.003914
118,Item2Vec_itemSim,epochs-160_learning_rate-0.25_negative_exp--0.5_subsample-0.001_w_size-5_epochs-140,0.003468
126,Item2Vec_itemSim,epochs-160_learning_rate-0.25_negative_exp--0.5_subsample-0.001_w_size-9999_epochs-140,0.003766
2268,Item2Vec_itemSim,epochs-160_learning_rate-0.25_negative_exp--1_subsample-0.001_w_size-3_epochs-140,0.005647


In [6]:
pd.options.display.max_colwidth = 100
df_display = saida[saida.Recommender != "ALS"][["Dataset", "Recommender", "Parameters", "NDCG@20"]]
df_display = df_display[saida.Recommender != "BPR"]
display(df_display)

C:\Users\Rafael Tofoli\AppData\Local\Temp\ipykernel_12372\2091895378.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_display = df_display[saida.Recommender != "BPR"]


,Dataset,Recommender,Parameters,NDCG@20
2,BestBuy,Item2Vec_itemSim,epochs-200_learning_rate-0.25_negative_exp--0.75_subsample-0.001_w_size-7_epochs-80,0.021500
3,BestBuy,TimeI2V_Cont,epochs-120_learning_rate-0.25_negative_exp--0.75_subsample-0.001_w_size--1_epochs-120,0.033796
4,BestBuy,TimeI2V_Disc,epochs-200_learning_rate-0.25_negative_exp--0.75_subsample-0.001_time_exp-1_w_size--1_epochs-120,0.028699
7,DeliciousBookmarks,Item2Vec_itemSim,epochs-160_learning_rate-0.025_negative_exp--0.75_subsample-0.0001_w_size--1_epochs-160,0.276692
8,DeliciousBookmarks,TimeI2V_Cont,epochs-120_learning_rate-0.25_negative_exp--1_subsample-0.001_w_size--1_epochs-40,0.258370
9,DeliciousBookmarks,TimeI2V_Disc,epochs-300_learning_rate-0.25_negative_exp--1_subsample-0.0001_time_exp-5_w_size--1_epochs-60,0.194031
12,MovieLens,Item2Vec_itemSim,epochs-120_learning_rate-0.25_negative_exp-1_subsample-0.001_w_size-5_epochs-40,0.050366
13,MovieLens,TimeI2V_Disc,epochs-160_learning_rate-0.25_negative_exp--0.75_subsample-0.001_time_exp-0_w_size-5_epochs-40,0.034670
16,RetailRocket-Transactions,Item2Vec_itemSim,epochs-120_learning_rate-0.025_negative_exp-0.75_subsample-0.0001_w_size-7_epochs-80,0.043697
17,RetailRocket-Transactions,TimeI2V_Cont,epochs-120_learning_rate-0.025_negative_exp-0.75_subsample-0.01_w_size-7_epochs-40,0.037296


# Análise de parâmetros - Item2Vec e Temporal Item2Vec

In [ ]:
'''new_df = metrics_df[metrics_df["Recommender"] == 'Item2Vec_itemSim']
new_df = new_df[new_df["Dataset"] ==  'BestBuy']

my_regex = "Recommender|Parameters|Dataset|" + "NDCG@20" + ".*"
new_df = new_df.filter(regex=(my_regex))

new_df[['Epochs', 'Factors', 'a', 'Learning_Rate', 'b', 'Negative_Exp', "c", "Negative_samples", 'Subsample',]] = new_df['Parameters'].str.split('_', expand=True)
new_df.drop(columns=['Parameters', 'a', 'b', 'c'], inplace=True)

new_df['Epochs'] = new_df['Epochs'].str.split('-', expand=True)[1]
new_df['Factors'] = new_df['Factors'].str.split('-', expand=True)[1]
new_df['Learning_Rate'] = new_df['Learning_Rate'].str.split('-', expand=True)[1]
new_df['Negative_samples'] = new_df['Negative_samples'].str.split('-', expand=True)[1]
new_df['Negative_Exp'] = new_df['Negative_Exp'].str[4:]
new_df['Subsample'] = new_df['Subsample'].str[10:]
new_df[["Epochs", 'Factors', 'Learning_Rate', "Negative_Exp", "Negative_samples", "Subsample"]] = new_df[["Epochs", 'Factors', 'Learning_Rate', "Negative_Exp", "Negative_samples", "Subsample"]].astype(float)


corr = new_df[['NDCG@20', "Epochs", 'Factors', 'Learning_Rate', "Negative_Exp", "Negative_samples", "Subsample"]].corr()
display(corr)


#sns.heatmap(corr)
#new_df.pivot_table(index=["Negative_Exp"], columns = ["Subsample"])'''

# Geração de gráficos

In [ ]:
def function_1(x):
    if(len(x.split('@')) > 1): 
        x = x.split('@')[1]
    return x

curr_metric = "Hit_Rate"

main_path = "results/metrics"
main_file = os.listdir(main_path)

for dataset_name in main_file:
    
    saida_aux = saida[saida['Dataset'] == dataset_name] 
        
    my_regex = "Recommender|" +  curr_metric + ".*"
    df_aux = saida_aux.filter(regex=(my_regex))
    
    df_aux = df_aux.set_index('Recommender')
    df_aux = df_aux.rename(columns=function_1)
    
    marker_symbols = ['circle', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagram']

    fig = go.Figure()

    for i, recomendador in enumerate(df_aux.index):
        fig.add_trace(
            go.Scatter(
                x=df_aux.columns,  # Colunas de métricas
                y=df_aux.loc[recomendador],  # Pontuações do recomendador
                mode='lines+markers',
                marker=dict(symbol=marker_symbols[i], size=7, line=dict(color='black', width=0.5)),
                name=recomendador
            )
        )
    
    fig.update_layout(
    title=dataset_name+" - "+curr_metric,
    title_x=0.15,
    title_y=0.8,
    showlegend=True,
    height=400,
    width=600,
    yaxis_title="",
    xaxis_title="",
    legend_title="Recommenders",
    font=dict(
        family="Helvetica",
        size=12,
        color="black"
        )
    )

    fig.show()

In [ ]:
'''def function_1(x):
    if len(x.split('@')) > 1: 
        x = x.split('@')[1]
    return x

curr_metric = "NDCG"

main_path = "results/metrics"
main_file = os.listdir(main_path)

# Determine the grid size
rows = 1
cols = 1
num_plots = len(main_file)

# Define colors for each recommender
color_map = {
    'ALS - UI': '#1f77b4',
    'ALS - II': '#ff7f0e',
    'ALS - WS': '#2ca02c',
    'BPR - UI': '#d62728',
    'BPR - II': '#9467bd',
    'BPR - WS': '#bcbd22',
}

# Create subplots with adjusted width and height
fig = make_subplots(
    rows=rows, 
    cols=cols, 
    subplot_titles=main_file,
    vertical_spacing=0.05,  # Decrease vertical space between plots
    horizontal_spacing=0.05  # Decrease horizontal space between plots
)

# Iterate over datasets and create subplots
for idx, dataset_name in enumerate(main_file):
    row = (idx // cols) + 1
    col = (idx % cols) + 1
    
    saida_aux = saida[saida['Dataset'] == dataset_name] 
    my_regex = "Recommender|" +  curr_metric + ".*"
    df_aux = saida_aux.filter(regex=(my_regex))
    
    df_aux["Recommender"] = df_aux["Recommender"].replace(["ALS", "ALS_itemSim", "ALS_weighted", "BPR", "BPR_itemSim", "BPR_weighted"], 
                                                  ["ALS - UI", "ALS - II", "ALS - WS", "BPR - UI", "BPR - II", "BPR - WS"])
        
    df_aux = df_aux.set_index('Recommender')
    df_aux = df_aux.rename(columns=function_1)
    
    marker_symbols = ['circle', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagram']

    for i, recomendador in enumerate(df_aux.index):
        fig.add_trace(
            go.Scatter(
                x=df_aux.columns,  # Colunas de métricas
                y=df_aux.loc[recomendador],  # Pontuações do recomendador
                mode='lines+markers',
                marker=dict(symbol=marker_symbols[i], size=7, color=color_map.get(recomendador, 'black'), line=dict(color='black', width=0.5)),
                name=recomendador,
                showlegend=(idx == 6)  # Show legend only for the specified subplot
            ),
            row=row,
            col=col
        )

fig.update_layout(
    height=1400,  # Increase height
    width=800,   # Decrease width
    title_text="",  # Global title
    showlegend=True,
    font=dict(
        family="Helvetica",
        size=12,
        color="black"
    ),
    legend_title="Recommenders",
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.04,  
        xanchor='center',
        x=0.5,
        font=dict(
            size=12.5
        )
    )
)

fig.show()'''

# Geração de tabelas - Divisão por Top-N

In [ ]:
# Gera uma tabela para todos os datasets em todos os Top-N diferentes
# Retorna o melhor parametro de cada dataset

def function_1(x):
   x = x.split('@')[0]
   return x

names_dict = {"Prec": "Precision", "Rec": "Recall", "F1_Score":"F1-Score", "Hit_Rate": "Hit-Rate"}

results_path = "latex_results"

string_final = ""

if not os.path.exists(results_path):
    os.makedirs(results_path)
    
main_file = os.listdir(main_path)

for dataset_name in main_file:
    
    saida_aux = saida[saida['Dataset'] == dataset_name] 
      
    string_final = string_final + ("=========== DATASET: " + dataset_name + " ===========\n\n\n")
    string_final = string_final + ("=========== MELHORES PARÂMETROS ===========\n\n")
    string_final = string_final + (saida[["Recommender", "Parameters"]].to_string())
    
    for curr_k in top_k:
        
        aux = '\n\n------ Top' + str(curr_k) + ' ------\n\n'
        string_final = string_final + (aux)

        #Pre processamento das colunas
        my_regex = "Recommender|" +  str(curr_k) + "$"
        df_aux = saida_aux.filter(regex=(my_regex))
        df_aux = df_aux.round(4)
        df_aux = df_aux.rename(columns=function_1)
        df_aux = df_aux.rename(columns=names_dict)

        # Tranforma a tabela em Latex
        capt = "Top " + str(curr_k) + " recommendation"
        string_final = string_final + (df_aux.style.hide(axis="index").set_caption(capt).to_latex())
        
        
    string_final = string_final + ("\n\n\n\n")

f = open(os.path.join(results_path, "saida.txt"), "w") 
f.write(string_final)
f.close()

# Geração de tabelas - Tabela unificada

In [ ]:
def pre_proc(df, comp_metric):
    
    #Seleciona apenas a metrica relevante
    df_aux = df.loc[:, ["Dataset", "Recommender", comp_metric]].copy()

    #Separa a coluna de recomendadores em dois
    df_aux[['Embedding', 'Recommender']]= df_aux['Recommender'].str.split('_', expand=True)

    #Preenche os valores nulos
    df_aux['Recommender'] = np.where(df_aux['Recommender'].isna() , df_aux['Embedding'], df_aux['Recommender'])
    
    #Renomeia colunas
    df_aux["Recommender"] = df_aux["Recommender"].replace(names_dict)
    
    df_pivot = df_aux.pivot(index='Dataset', columns=['Embedding', 'Recommender'], values=comp_metric)
    df_pivot = df_pivot.reset_index()
    
    return df_pivot

In [ ]:
def create_gray_scale(df_pivot):
    
    #Cria a escala de cinza
    GS = pd.concat([
        pd.DataFrame(MinMaxScaler(feature_range=(0.5, 0.95)).fit_transform(-df_pivot.iloc[:,1:4].values.T).round(4).T),
        pd.DataFrame(MinMaxScaler(feature_range=(0.5, 0.95)).fit_transform(-df_pivot.iloc[:,4:].values.T).round(4).T),
    ], axis=1)
    GS.index = df_pivot.index
    GS.columns = df_pivot.iloc[:,1:].columns
    
    return GS

In [ ]:
def add_gain(df_optimal, df_real):
    
    real_metrics = df_real.values[:,1:]
    optimal_metrics = df_optimal.values[:,1:]
    
    gain_percentage = (((optimal_metrics - real_metrics)/real_metrics)*100).astype(float).round(1).astype(str)
    
    df_real.iloc[:,2:4] = df_real.round(4).astype(str).iloc[:,2:4] + " (+" +  gain_percentage[:,1:3] + "\%)"
    df_real.iloc[:,5:] = df_real.round(4).astype(str).iloc[:,5:] + " (+" +  gain_percentage[:,4:] + "\%)"
    
    return df_real

In [ ]:
#comp_metric = "NDCG@10"

f = open(os.path.join(results_path, "saida_tabela_unificada.txt"), "w")

names_dict = {"itemSim": "Item-Item", "weighted": "Weighted", "ALS": "User-Item", "BPR": "User-Item"}

#Seleciona apenas as colunas terminadas em @10
metrics_names = saida.filter(regex='@10').columns

for comp_metric in metrics_names:
        
    df_final = pre_proc(saida, comp_metric)
    GS = create_gray_scale(df_final)
    
    #df_final = add_gain(df_optimal, df_real)
    
    #Coloca a escala na tabela
    df_final.iloc[:,1:] = "\\cellcolor[gray]{" + GS.astype(str) + "}" + df_final.round(4).astype(str).iloc[:,1:]
        
    tabela_latex = (df_final.style.hide(axis="index")
                    .to_latex(column_format='lccc|ccc', 
                              multicol_align='c',
                              hrules='True',
                              position_float='centering',
                              caption="Comparação de diversos métodos sob " + comp_metric))

    #Centraliza a tabela
    tabela_latex = tabela_latex.replace("\\begin{tabular}", "\\resizebox{\\textwidth} {!}{ \\begin{tabular}")
    tabela_latex = tabela_latex.replace("\\end{tabular}", "\\end{tabular} }")

    results_path = "latex_results"

    if not os.path.exists(results_path):
        os.makedirs(results_path)

    f.write("\n\n\n------------------  " + "MÉTRICA: " + comp_metric + " ------------------\n\n\n")
    f.write(tabela_latex)

f.close()

# Geração de tabelas - Comparação Item2Vec - TemporalI2V

In [ ]:
f = open(os.path.join(results_path, "saida_tabela_comparacao.txt"), "w")

comp_metric = "NDCG@10"
metrics_names = saida.filter(regex=comp_metric).columns

df_aux = saida.loc[:, ["Dataset", "Recommender", comp_metric]].copy()
df_aux = df_aux[df_aux['Recommender'] != 'ALS']
df_aux = df_aux[df_aux['Recommender'] != 'BPR']

df_pivot = df_aux.pivot(index='Dataset', columns='Recommender', values=comp_metric)
df_pivot = df_pivot.rename_axis(None, axis=1).reset_index()

for column in df_pivot.columns:
    if(column == 'Item2Vec_itemSim' or column == 'Dataset'):
        continue
    df_pivot[column] = round((df_pivot[column] - df_pivot['Item2Vec_itemSim'])/df_pivot['Item2Vec_itemSim'] * 100, 2)

df_pivot = df_pivot.drop(columns=['Item2Vec_itemSim'])
df_pivot.iloc[:,1:] = df_pivot.iloc[:,1:].astype(str) + '\%'

tabela_comparacao = df_pivot.style.hide(axis="index").to_latex()
tabela_comparacao = tabela_comparacao.replace("_", "-")

print(tabela_comparacao)

f.close()